# Deep Learning IDS Using LSTM — Google Colab

**Project:** Design and Implementation of a Deep Learning Intrusion Detection System Using LSTM
**Author:** Kayode Timileyin Nicholas | FUTA Cybersecurity Final Year Project

---

### How to use
1. Upload `lstm_project.zip` (the project code) to Google Drive (once)
2. Upload `lstm_raw.zip` (the datasets) to Google Drive (once)
3. Run all cells top-to-bottom
4. Results are saved to `outputs/<dataset>/` and backed up to Drive

### What this notebook does
- Extracts your project from a zip on Google Drive (no GitHub auth needed)
- Installs all dependencies
- Runs the full pipeline for each dataset (NSL-KDD, CICIDS2017, UNSW-NB15)
- Backs up results to Google Drive under `MyDrive/lstm_ids_results/<dataset>/`

---
## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BACKUP = "/content/drive/MyDrive/lstm_ids_results"
!mkdir -p {DRIVE_BACKUP}
print(f"Backup folder: {DRIVE_BACKUP}")

---
## Cell 2 — Extract project code from Drive

You should have uploaded `lstm_project.zip` to your Google Drive.

This cell searches for it and extracts the project code.

In [ ]:
import os

# Check if we're already in the project
if os.path.exists("run_pipeline.py") and os.path.exists("config.yaml"):
    print("Project code already present. Skipping extraction.")
else:
    # Search common Drive locations
    search_paths = [
        "/content/drive/MyDrive/lstm_project.zip",
        "/content/drive/MyDrive/lstm_ids_project.zip",
    ]
    zip_path = None
    for p in search_paths:
        if os.path.exists(p):
            zip_path = p
            break

    if zip_path is None:
        # Brute-force search
        print("Zip not in expected locations, searching Drive...")
        results = !find /content/drive -name "lstm_project.zip" -o -name "lstm_ids_project.zip" 2>/dev/null | head -5
        if results:
            zip_path = results[0]

    if zip_path:
        print(f"Found project zip: {zip_path}")
        print(f"Size: {os.path.getsize(zip_path) / 1e6:.1f} MB")
        print("Extracting project code...")
        !unzip -q -o "{zip_path}" -d /content/
        print(f"Extracted to: {os.getcwd()}")
    else:
        print("ERROR: lstm_project.zip not found on Google Drive.")
        print("Please upload it to MyDrive/ and re-run this cell.")
        print("\nTo create the zip from your local machine, run:")
        print("  cd /path/to/lstm_ids_project")
        print("  bash zip_project.sh")

---
## Cell 3 — Rename local tensorflow/ shim (if present)

Your project may have a local `tensorflow/` directory that routes imports to PyTorch.
Colab has real TensorFlow with GPU — we need to use it instead.

In [ ]:
# Rename the local tensorflow shim so real TF is used
if os.path.exists("tensorflow"):
    !mv tensorflow tensorflow_shim_local
    print("Renamed tensorflow/ shim to tensorflow_shim_local/")
else:
    print("No tensorflow/ shim found (already renamed or not present).")

# Force TensorFlow backend for Keras 3
os.environ['KERAS_BACKEND'] = 'tensorflow'

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

---
## Cell 4 — Install dependencies

In [ ]:
# Install only what Colab doesn't already have
!pip install -q -r requirements_colab.txt

# Also install click (used by the pipeline CLI)
!pip install -q click

print("\nAll dependencies installed.")

---
## Cell 5 — Extract dataset from Google Drive

**First run only.** You should have uploaded `lstm_raw.zip` to Google Drive.

Expected location: `MyDrive/lstm_raw.zip`

This cell finds the zip and extracts it into `data/raw/`.

In [ ]:
LOCAL_DATA = "data/raw"

# Check if data already extracted
if os.path.exists(f"{LOCAL_DATA}/nsl_kdd") and os.path.exists(f"{LOCAL_DATA}/unsw_nb15"):
    print("Data already present locally. Skipping extraction.")
else:
    # Search common Drive locations for the zip
    search_paths = [
        "/content/drive/MyDrive/lstm_raw.zip",
        "/content/drive/MyDrive/lstm_ids_project/data/raw/lstm_raw.zip",
    ]
    zip_path = None
    for p in search_paths:
        if os.path.exists(p):
            zip_path = p
            break

    if zip_path is None:
        # Brute-force search (slow but guaranteed)
        print("Zip not in expected locations, searching Drive...")
        results = !find /content/drive -name "lstm_raw.zip" -type f 2>/dev/null
        if results:
            zip_path = results[0]

    if zip_path:
        print(f"Found zip: {zip_path}")
        print(f"Size: {os.path.getsize(zip_path) / 1e6:.1f} MB")
        print("Extracting (this may take 1-3 minutes)...")
        !mkdir -p {LOCAL_DATA}
        !unzip -q -o "{zip_path}" -d {LOCAL_DATA}
        print("Extraction complete.")
    else:
        print("ERROR: lstm_raw.zip not found on Google Drive.")
        print("Please upload it to MyDrive/ and re-run this cell.")

# Show what we have
!du -sh {LOCAL_DATA}/*

---
## Cell 6 — Verify environment

In [ ]:
import sys
import numpy as np
import keras

print(f"Python:     {sys.version}")
print(f"NumPy:      {np.__version__}")
print(f"Keras:      {keras.__version__}")
print(f"Backend:    {keras.backend.backend()}")

# GPU check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU:        {gpus[0].name}")
else:
    print("GPU:        NONE (using CPU)")

# Check project structure
print(f"\nProject files:")
!ls run_pipeline.py config.yaml requirements_colab.txt 2>&1

# Check data
print(f"\nDataset directories:")
!ls data/raw/

---
## Cell 7 — Enable GPU memory growth

Prevents TensorFlow from grabbing all GPU VRAM at once.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU memory growth enabled for {len(gpus)} device(s)")
else:
    print("No GPU detected — running on CPU")

---
## Cell 8 — Run Pipeline: NSL-KDD (primary dataset)

Takes ~15-20 min on T4 GPU.

In [ ]:
import shutil

os.environ['KERAS_BACKEND'] = 'tensorflow'

print("Running NSL-KDD pipeline...")
print("=" * 60)
!python run_pipeline.py --dataset nsl_kdd 2>&1 | tail -80

---
## Cell 9 — Backup NSL-KDD results to Drive

In [ ]:
import shutil

backup = f"{DRIVE_BACKUP}/nsl_kdd"
os.makedirs(backup, exist_ok=True)

# Backup per-dataset outputs
ds_out = "outputs/nsl_kdd"
if os.path.exists(ds_out):
    shutil.copytree(ds_out, f"{backup}/outputs", dirs_exist_ok=True)
    print(f"Backed up outputs to {backup}/outputs/")

# Also backup the flat directories (baselines, models) as fallback
for d in ["models/baselines", "models/final"]:
    if os.path.exists(d):
        shutil.copytree(d, f"{backup}/{d}", dirs_exist_ok=True)

print(f"\nNSL-KDD backup complete: {backup}")
!find {backup} -type f | head -30

---
## Cell 10 — Run Pipeline: UNSW-NB15

Large dataset — uses `--subsample 0.3` (30% stratified sample).
Takes ~30-45 min on T4 GPU.

In [ ]:
import shutil

os.environ['KERAS_BACKEND'] = 'tensorflow'

print("Running UNSW-NB15 pipeline (subsample 0.3)...")
print("=" * 60)
!python run_pipeline.py --dataset unsw_nb15 --subsample 0.3 2>&1 | tail -80

---
## Cell 11 — Backup UNSW-NB15 results to Drive

In [ ]:
import shutil

backup = f"{DRIVE_BACKUP}/unsw_nb15"
os.makedirs(backup, exist_ok=True)

ds_out = "outputs/unsw_nb15"
if os.path.exists(ds_out):
    shutil.copytree(ds_out, f"{backup}/outputs", dirs_exist_ok=True)
    print(f"Backed up outputs to {backup}/outputs/")

for d in ["models/baselines", "models/final"]:
    if os.path.exists(d):
        shutil.copytree(d, f"{backup}/{d}", dirs_exist_ok=True)

print(f"\nUNSW-NB15 backup complete: {backup}")
!find {backup} -type f | head -30

---
## Cell 12 — Run Pipeline: CICIDS2017 (largest dataset)

Takes ~45-60 min on T4 GPU.

In [ ]:
import shutil

os.environ['KERAS_BACKEND'] = 'tensorflow'

print("Running CICIDS2017 pipeline...")
print("=" * 60)
!python run_pipeline.py --dataset cicids2017 2>&1 | tail -80

---
## Cell 13 — Backup CICIDS2017 results to Drive

In [ ]:
import shutil

backup = f"{DRIVE_BACKUP}/cicids2017"
os.makedirs(backup, exist_ok=True)

ds_out = "outputs/cicids2017"
if os.path.exists(ds_out):
    shutil.copytree(ds_out, f"{backup}/outputs", dirs_exist_ok=True)
    print(f"Backed up outputs to {backup}/outputs/")

for d in ["models/baselines", "models/final"]:
    if os.path.exists(d):
        shutil.copytree(d, f"{backup}/{d}", dirs_exist_ok=True)

print(f"\nCICIDS2017 backup complete: {backup}")
!find {backup} -type f | head -30

---
## Cell 14 — Summary of all runs

In [ ]:
import json
from pathlib import Path

print("=" * 60)
print("RESULTS SUMMARY — All Datasets")
print("=" * 60)

for ds in ["nsl_kdd", "unsw_nb15", "cicids2017"]:
    # Look in per-dataset outputs first, then Drive backup
    candidates = [
        Path(f"outputs/{ds}/metrics/evaluation_results.json"),
        Path(f"{DRIVE_BACKUP}/{ds}/outputs/metrics/evaluation_results.json"),
        Path(f"{DRIVE_BACKUP}/{ds}/reports/metrics/evaluation_results.json"),
    ]
    result_file = None
    for c in candidates:
        if c.exists():
            result_file = c
            break

    print(f"\n--- {ds.upper()} ---")
    if result_file:
        print(f"  Source: {result_file}")
        with open(result_file) as f:
            data = json.load(f)
        for model_name, metrics in data.items():
            acc = metrics.get("accuracy", "N/A")
            f1 = metrics.get("f1_macro", "N/A")
            roc = metrics.get("roc_auc_macro", "N/A")
            if isinstance(acc, float):
                print(f"  {model_name:25s}  Acc={acc:.4f}  F1={f1:.4f}  ROC={roc:.4f}")
            else:
                print(f"  {model_name:25s}  Acc={acc}  F1={f1}  ROC={roc}")
    else:
        print("  No results found.")

---
## Cell 15 — Download all results as a ZIP

In [ ]:
!zip -r /content/all_results.zip outputs/ 2>/dev/null
print("ZIP created. Download from the Colab file browser:")
print("  Left sidebar \u2192 Files \u2192 all_results.zip \u2192 \u22ee \u2192 Download")